In [64]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [ ]:
from pathlib import Path
import sys

root = Path.cwd().resolve()
root = root.parent
sys.path.insert(0, str(root))

import torch
import math
from torch.utils.data import Dataset, DataLoader, TensorDataset, random_split
from transformers import AutoTokenizer, AutoModelForCausalLM
from typing import List, Dict, Optional
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from datasets import load_dataset
from sae_trainer.core.model_utils import ReluSparseAutoencoder, TopKSparseAutoencoder, sae_inserted_llm
from sae_trainer.core.model_utils import sae_steered_llm


In [49]:
device = "cuda" if torch.cuda.is_available() else "cpu"
dataset_name = 'openwebtext'
model_name_simple = "qwen" # Either gpt2 or qwen
sae_type = 'topk'

if model_name_simple == "gpt2":
    model_name = "gpt2"
elif model_name_simple == "qwen":
    model_name = "Qwen/Qwen2-0.5B-Instruct"
else:
    raise ValueError(f"Invalid model name: {model_name_simple}. Expected either 'gpt2' or 'qwen'.")
tokenizer = AutoTokenizer.from_pretrained(model_name)
llm = AutoModelForCausalLM.from_pretrained(model_name).to(device).eval()

def get_sae(sae_type, model_name_simple, layer_idx, dataset_name, device):
    k = 40

    # Load SAE checkpoint — pick whichever layer you want to analyse (3, 6, 9, or 11)
    ckpt = torch.load(root / f"model_weights_files/{sae_type}_sae_{model_name_simple}_{dataset_name}_layer{layer_idx}.pt", map_location=device)
    if sae_type == 'relu':
        sae = ReluSparseAutoencoder(d_in=ckpt["d_in"], d_latent=ckpt["d_latent"], normalize_decoder=True).to(device)
    elif sae_type == 'topk':
        sae = TopKSparseAutoencoder(d_in=ckpt["d_in"], d_latent=ckpt["d_latent"], k=k, normalize_decoder=True).to(device)
    else:
        raise ValueError(f"Invalid sae type: {sae_type}. Expected either 'relu' or 'qwen'.")
    sae.load_state_dict(ckpt["model_state"])
    sae.eval()

    return sae

In [ ]:
# Build openwebtext dataset and FeatureTracer summaries. Keep defaults to reproduce published results

n_train_docs = 4000 # cfg.max_batches * cfg.collection_batch_size

ds = load_dataset("openwebtext", split="train", streaming=True)
# Exclude training data
inputs = ds.skip(n_train_docs)

eval_texts = [
    " ".join(row["text"].split()[:150])   # truncate to 150 words
    for row in inputs.shuffle(seed=42, buffer_size=1000)
    if len(row["text"].split()) >= 50     # skip very short examples
][:100]

transformers.models.qwen2.modeling_qwen2.Qwen2ForCausalLM

In [50]:
# testing sae_inserted_llm hook

sae = get_sae(sae_type=sae_type, model_name_simple=model_name_simple, layer_idx=12, dataset_name=dataset_name, device=device)

inputs = tokenizer("The capital of France is", return_tensors="pt").to(device)

with sae_inserted_llm(llm, sae, layer_idx=6, device=device):
    out = llm.generate(**inputs, max_new_tokens=20)

print(tokenizer.decode(out[0], skip_special_tokens=True))


The capital of France is not a city but it has been the center for the country. In 1806,


In [47]:
# Use perplexity to evaluate how sae affects llm confidence 
@torch.no_grad()
def compute_perplexity(llm, tokenizer, texts: list[str], device, max_length=512) -> float:
    """Mean perplexity over a list of texts."""
    llm.eval()
    total_loss, total_tokens = 0.0, 0

    for text in texts:
        inputs = tokenizer(
            text, return_tensors="pt", truncation=True, max_length=max_length
        ).to(device)
        if inputs["input_ids"].shape[1] < 2:
            continue
        loss = llm(**inputs, labels=inputs["input_ids"]).loss
        n_tokens = inputs["input_ids"].shape[1] - 1  # predicting next token
        total_loss  += loss.item() * n_tokens
        total_tokens += n_tokens

    return math.exp(total_loss / total_tokens)


In [52]:
def get_perplexity_metrics(layer_idx, sae, llm, tokenizer, inputs, device):
    ppl_baseline = compute_perplexity(llm, tokenizer, inputs, device)

    with sae_inserted_llm(llm, sae, layer_idx=layer_idx, device=device):
        ppl_sae = compute_perplexity(llm, tokenizer, inputs, device)

    # CE loss recovered — fraction of baseline predictive performance preserved
    # 1.0 = perfect reconstruction, 0.0 = SAE adds as much loss as the whole model
    ce_baseline = math.log(ppl_baseline)
    ce_sae = math.log(ppl_sae)
    ce_loss_recovered =  1 - (ce_sae - ce_baseline) / ce_baseline

    out = {
        'Baseline_PPL': ppl_baseline,
        'SAE_Inserted_PPL': ppl_sae,
        'Delta_PPL': ppl_sae - ppl_baseline,
        'CE_loss_recovered': ce_loss_recovered
    }

    return out


In [62]:
def get_perplexity_metrics_from_steered_llm(layer_idx, feature_id, delta, sae, llm, tokenizer, inputs, device):
    ppl_baseline = compute_perplexity(llm, tokenizer, inputs, device)

    with sae_steered_llm(llm, sae, layer_idx=layer_idx, feature_id=feature_id, delta=delta):
        ppl_sae = compute_perplexity(llm, tokenizer, inputs, device)

    # CE loss recovered — fraction of baseline predictive performance preserved
    # 1.0 = perfect reconstruction, 0.0 = SAE adds as much loss as the whole model
    ce_baseline = math.log(ppl_baseline)
    ce_sae = math.log(ppl_sae)
    ce_loss_recovered =  1 - (ce_sae - ce_baseline) / ce_baseline

    out = {
        'Baseline_PPL': ppl_baseline,
        'SAE_Inserted_PPL': ppl_sae,
        'Delta_PPL': ppl_sae - ppl_baseline,
        'CE_loss_recovered': ce_loss_recovered
    }

    return out

In [ ]:
layer_idxs = [12, 16]
out = {}

for layer_idx in layer_idxs:
    out[layer_idx] = get_perplexity_metrics(layer_idx, sae, llm, tokenizer, inputs, device)


In [54]:
for layer_idx in out.keys():
    print(out[layer_idx])

{'Baseline_PPL': 59629.36851310031, 'SAE_Inserted_PPL': 22137775.82245145, 'Delta_PPL': 22078146.45393835, 'CE_loss_recovered': 0.46190028560610863}
{'Baseline_PPL': 59629.36851310031, 'SAE_Inserted_PPL': 1481419.5664274353, 'Delta_PPL': 1421790.197914335, 'CE_loss_recovered': 0.7078359351816674}


In [ ]:
# Next - find interesting SAE features to up/downweight. Compare perplexity metrics after doing this
# Also, possibly train more qwen SAEs to try improving perplexity

In [59]:
# Exp 1: Ask the LLM something un-related to the feature being manipulated

inputs = tokenizer("The Eiffel Tower is located in", return_tensors="pt").to(device)

layer_idx = 12
feature_id = 6899

# Control: no sae feature manipulation
out = llm.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(out[0], skip_special_tokens=True))

# Amplify feature 'feature_id'
with sae_steered_llm(llm, sae, layer_idx=layer_idx, feature_id=feature_id, delta=10.0):
    out = llm.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(out[0], skip_special_tokens=True))

# Suppress feature 'feature_id' (large negative delta drives it to 0 via the clamp)
with sae_steered_llm(llm, sae, layer_idx=layer_idx, feature_id=feature_id, delta=-100.0):
    out = llm.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(out[0], skip_special_tokens=True))


The Eiffel Tower is located in Paris, France. The tower stands 324 meters (1067 feet) tall
The Eiffel Tower is located in the 1974, and the French team scored the goal. The second was a nice
The Eiffel Tower is located in the city of Paris, and it’s a place where I’ve always been obsessed with. It’s


In [61]:
# Exp 2: Ask the LLM something it needs to use the manipulated feature to answer

inputs = tokenizer("My team needs one more goal to win the game, and I have the ball. I am going to", return_tensors="pt").to(device)

layer_idx = 12
feature_id = 6899

# Control: no sae feature manipulation
out = llm.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(out[0], skip_special_tokens=True))

# Amplify feature 'feature_id'
with sae_steered_llm(llm, sae, layer_idx=layer_idx, feature_id=feature_id, delta=10.0):
    out = llm.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(out[0], skip_special_tokens=True))

# Suppress feature 'feature_id' (large negative delta drives it to 0 via the clamp)
with sae_steered_llm(llm, sae, layer_idx=layer_idx, feature_id=feature_id, delta=-100.0):
    out = llm.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(out[0], skip_special_tokens=True))

My team needs one more goal to win the game, and I have the ball. I am going to try to get it by dribbling around the corner of the court, but the defense is too strong
My team needs one more goal to win the game, and I have the ball. I am going to score a 20 yard goal for the winner. The other two were in on by the goalie
My team needs one more goal to win the game, and I have the ball. I am going to be a 20 year old. The 1983-1984 was


In [63]:


out = get_perplexity_metrics_from_steered_llm(layer_idx, feature_id, 10, sae, llm, tokenizer, inputs, device)
out

{'Baseline_PPL': 59629.36851310031,
 'SAE_Inserted_PPL': 16826184.090376683,
 'Delta_PPL': 16766554.721863583,
 'CE_loss_recovered': 0.48685041479255864}

In [65]:
# Exp 3: Manipulate a "punctuation feature" with a random prompt

inputs = tokenizer("The Eiffel Tower is located in", return_tensors="pt").to(device)

layer_idx = 12
feature_id = 348

# Control: no sae feature manipulation
out = llm.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(out[0], skip_special_tokens=True))

# Amplify feature 'feature_id'
with sae_steered_llm(llm, sae, layer_idx=layer_idx, feature_id=feature_id, delta=10.0):
    out = llm.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(out[0], skip_special_tokens=True))

# Suppress feature 'feature_id' (large negative delta drives it to 0 via the clamp)
with sae_steered_llm(llm, sae, layer_idx=layer_idx, feature_id=feature_id, delta=-100.0):
    out = llm.generate(**inputs, max_new_tokens=20)
print(tokenizer.decode(out[0], skip_special_tokens=True))

The Eiffel Tower is located in Paris, France. It was built by Gustave Eiffel and opened to the public on May
The Eiffel Tower is located in the city of the United Kingdom, where a small, 100-year-old French architect,
The Eiffel Tower is located in Paris, France. The Eiffaut is a massive, 100 feet of white concrete
